# Post-Processing of the Results

This notebook turns the results of the main experiment as they were recorded (`results/main` at the `original-results` tag) into the results used in the analysis.

The process consists of:
1. **Loading**: Reading each run and restoring the originally recorded values.
2. **Counter Overflow Correction**: Reconstructing the CPU and DRAM energy lost to the wrong handling of the RAPL counter overflow.
3. **Run Summaries**: Recomputing `run_data.json` and the run table with the current definitions of `evaluate_from_local.py`.
4. **Applying**: Writing the per-prompt records, the summaries, the run table and a log of the corrections.
5. **Validation**: Checking the corrected values and the recomputed summaries.

The notebook can be run any number of times, on the original results or on results that were already processed, and always produces the same files. GPU energy is never modified. The CPU and DRAM values cover the first of the server's two processors only: the second one was not measured and is not estimated.

### Setup & Imports
With `APPLY = False`, nothing is written: the notebook only reports what would change.

In [1]:
import csv
import json
import math
import statistics
import sys
from pathlib import Path

import pandas as pd

ROOT_DIR = Path("..").resolve()

# Summary of a run, as computed by evaluate_from_local.py at the end of a run
sys.path.append(str(ROOT_DIR / "dependencies/MMLU-Pro"))
from run_summary import summarize_run

# Results of the main experiment, processed in place
RESULTS_DIR = ROOT_DIR / "results/main"
# Write the processed results to disk
APPLY = True

# Size the script wrongly assumed for the RAPL counters (µJ)
M = 2**32
# Actual size of the RAPL counters of the server (µJ), from /sys/class/powercap/intel-rapl:*/max_energy_range_uj
MAX_ENERGY_RANGE_UJ = 262143328850

# Measured fields of each per-prompt record
CPU_FIELD = "inference_cpu_joules"
DRAM_FIELD = "inference_dram_joules"
GPU_FIELD = "inference_gpu_joules"
TOTAL_FIELD = "total_inference_joules"
TIME_FIELD = "inference_time_seconds"

# Field added by the correction: the originally recorded value and what was restored
CORRECTION_FIELD = "rapl_correction"

### 1. Loading the Runs
We read the per-prompt records and the summary of each run. Corrected records keep their originally recorded value in the `rapl_correction` field, from which they are restored before the correction is applied again.

In [2]:
def run_directories():
    directories = [d for d in RESULTS_DIR.glob("run_*") if d.is_dir()]
    # run_<model index>_repetition_<repetition index>
    return sorted(directories, key=lambda d: tuple(int(x) for x in d.name.split("_")[1::2]))


def load_run(run_dir):
    # Returns {file path: records} with the per-prompt records of a run, and its summary
    files = {}
    for path in sorted(run_dir.glob("*/CoT/all/*.json")):
        with open(path, "r") as f:
            files[path] = json.load(f)
    with open(run_dir / "run_data.json", "r") as f:
        run_data = json.load(f)
    return files, run_data


def restore_recorded_values(record):
    correction = record.pop(CORRECTION_FIELD, {})
    if "cpu" in correction:
        record[CPU_FIELD] = correction["cpu"]["stored_joules"]
    if "dram" in correction:
        record[DRAM_FIELD] = correction["dram"]["stored_joules"]


    record[TOTAL_FIELD] = record[CPU_FIELD] + record[DRAM_FIELD] + record[GPU_FIELD]


print(f"Runs found: {len(run_directories())}")

Runs found: 530


### 2. Counter Overflow Correction
The RAPL counters restart from zero after $R = 262{,}143{,}328{,}850$ µJ (about 262 kJ). The script used in the experiment computed the energy of an inference as `(end - start) % 2**32`, assuming a range of $M = 2^{32}$ µJ (about 4,295 J). Inferences that used more than 4,295 J therefore lost one or more multiples of $M$.

For a recorded value $s$, the true energy is one of:

$$\text{true} = s + k \cdot M \qquad \text{(no overflow during the inference)}$$
$$\text{true} = s + (R \bmod M) + k \cdot M \qquad \text{(one overflow during the inference)}$$

- **Finding $k$**: The CPU power is very stable within a run (about 1%), so we take the $k$ that brings the energy closest to the reference power of the run multiplied by the inference time.
- **Overflow case**: Only assumed when the recorded value is impossible otherwise (a short inference recorded with more than 4,000 J). Undetected overflows leave a value about 150 J too low, at most once per 262 kJ consumed.
- **Reference power of a run**: The power that agrees with the largest number of recorded values modulo $M$.

In [3]:
R_MOD_M = MAX_ENERGY_RANGE_UJ % M


def reference_power(records, field):
    # Recorded energy (µJ) and time of every inference long enough to give a stable power reading
    timed = [(r[field] * 1e6, r[TIME_FIELD]) for r in records if r[TIME_FIELD] >= 1]

    def distance(s, t, power):
        d = (power * t * 1e6 - s) % M
        return min(d, M - d)

    def agreeing(power):
        return [(s, t) for s, t in timed if distance(s, t, power) < 0.05 * power * t * 1e6 + 20e6]

    # Candidate powers: the ones given by the recorded values themselves, rounded to 0.5 W
    candidates = {round(s / t / 1e6 * 2) / 2 for s, t in timed if s > 0}
    best = max(candidates, key=lambda power: len(agreeing(power)))

    # Median power of the agreeing values, after restoring their lost multiples of M
    return statistics.median((s + round((best * t * 1e6 - s) / M) * M) / t / 1e6 for s, t in agreeing(best))


def correct_overflow(recorded_joules, time_seconds, power):
    s = round(recorded_joules * 1e6)
    expected = power * time_seconds * 1e6

    # The counter did not overflow: restore the k lost multiples of M
    k = max(0, round((expected - s) / M))
    value, overflowed = s + k * M, False
    error = abs(value - expected)

    # The counter overflowed during an inference of less than R mod M: the recorded value is about M too high
    overflow_value = s + R_MOD_M - M
    if error > 0.5 * M and overflow_value >= 0 and abs(overflow_value - expected) < 0.1 * error:
        value, overflowed, k = overflow_value, True, -1
        error = abs(value - expected)

    return {
        "joules": value / 1e6,
        "k": k,
        "overflowed": overflowed,
        "expected_joules": expected / 1e6,
        # Distance to the expected value, as a fraction of M. A wrong k would be about 1.
        "error_fraction": error / M,
    }


def apply_overflow_correction(records):
    # Corrects the CPU and DRAM energy of every record of a run. Returns the corrections made.
    log = []
    powers = {"cpu": reference_power(records, CPU_FIELD), "dram": reference_power(records, DRAM_FIELD)}

    for record in records:
        correction = {}
        for domain, field in (("cpu", CPU_FIELD), ("dram", DRAM_FIELD)):
            result = correct_overflow(record[field], record[TIME_FIELD], powers[domain])
            if result["k"] != 0:
                correction[domain] = {"stored_joules": record[field], "wrapped": int(result["overflowed"]), "k": result["k"]}
                log.append({
                    "question_id": record["question_id"],
                    "domain": domain,
                    "inference_time_seconds": record[TIME_FIELD],
                    "stored_joules": record[field],
                    "corrected_joules": result["joules"],
                    "expected_joules": round(result["expected_joules"], 3),
                    "wrapped": int(result["overflowed"]),
                    "k": result["k"],
                    "error_fraction": round(result["error_fraction"], 3),
                })
                record[field] = result["joules"]

        if correction:
            record[TOTAL_FIELD] = record[CPU_FIELD] + record[DRAM_FIELD] + record[GPU_FIELD]
            # Keeps the originally recorded values, so the correction can be traced and reverted
            record[CORRECTION_FIELD] = correction

    return log, powers

### 3. Run Summaries
We recompute the summary of each run with `summarize_run` (`dependencies/MMLU-Pro/run_summary.py`), the function that `evaluate_from_local.py` calls at the end of a run. Compared with the original results, the efficiency metrics use the GPU energy only and the metrics per token count the generated tokens only.

| | Fields |
|---|---|
| Removed | `accuracy_per_joule`, `tokens_per_joule`, `joules_per_token`, `avg_joules_per_inference` |
| Added | `accuracy_per_gpu_joule`, `gpu_joules_per_correct_answer`, `generated_tokens_per_gpu_joule`, `gpu_joules_per_generated_token`, `avg_gpu_joules_per_inference`, `median_gpu_joules_per_inference`, `avg_generated_tokens_per_inference`, `median_generated_tokens_per_inference`, `truncated_responses`, `unparsed_responses`, `median_time_per_inference_seconds` |

The accuracy is kept as recorded. The limit of generated tokens, needed to count the truncated responses, is 2,048 tokens plus 4,096 for reasoning models. The three Ministral 3 instruct models of this execution are no longer in `selected_models.csv` and are looked up through their 16-bit versions.

In [4]:
# Limit of generated tokens of a run: the default of evaluate_from_local.py, plus the thinking budget
# that experiments/main/RunnerConfig.py gives to reasoning models
MAX_NEW_TOKENS = 2048
MAX_THINKING_TOKENS = 4096

models = pd.read_csv(ROOT_DIR / "model-selection/selected_models.csv").set_index("model")

# Models of this execution that were later replaced in the selection: the Ministral 3 instruct
# models ran with 8-bit (FP8) weights, and the selection now lists their 16-bit versions
SUPERSEDED_MODELS = {
    f"mistralai/Ministral-3-{size}-Instruct-2512": f"mistralai/Ministral-3-{size}-Instruct-2512-BF16"
    for size in ("3B", "8B", "14B")
}


def max_new_tokens(model):
    model = SUPERSEDED_MODELS.get(model, model)
    return MAX_NEW_TOKENS + (MAX_THINKING_TOKENS if models.loc[model, "is_thinking"] else 0)


def recompute_summary(records, run_data):
    limit = max_new_tokens(run_data["model"])
    if max(r["generated_tokens"] for r in records) > limit:
        raise ValueError(f"{run_data['model']}: a response is longer than the limit of {limit} tokens.")
    return summarize_run(records, run_data["model"], run_data["avg_accuracy"], limit)

### 4. Applying the Post-Processing
We process one run at a time: restore the recorded values, apply the correction and recompute the summary. Files are only rewritten when their content changes. When `APPLY` is `True`, the notebook writes the per-prompt records, the `run_data.json` files, `run_table.csv` and `rapl_corrections.csv`.

In [5]:
inference_rows = []       # one row per inference
run_rows = []             # one row per run
overflow_log = []         # one row per corrected value
new_summaries = {}        # run name -> recomputed summary
previous_summaries = {}   # run name -> summary found on disk
files_changed = 0
summaries_changed = 0

for run_dir in run_directories():
    files, run_data = load_run(run_dir)
    records = [record for file_records in files.values() for record in file_records]
    if not records:
        print(f"[skip] {run_dir.name}: no per-prompt records")
        continue

    # Content currently on disk, to detect which files change
    on_disk = {path: json.dumps(file_records) for path, file_records in files.items()}

    # 1. Restore the recorded values
    for record in records:
        restore_recorded_values(record)
    recorded = {"cpu": sum(r[CPU_FIELD] for r in records), "total": sum(r[TOTAL_FIELD] for r in records)}
    recorded_cpu = [r[CPU_FIELD] for r in records]

    # 2. Reconstruct the energy lost to the counter overflow
    log, powers = apply_overflow_correction(records)
    for entry in log:
        overflow_log.append({"run": run_dir.name, **entry})

    # 3. Recompute the summary of the run
    summary = recompute_summary(records, run_data)
    new_summaries[run_dir.name] = summary
    previous_summaries[run_dir.name] = run_data

    for record, recorded_joules in zip(records, recorded_cpu):
        inference_rows.append({
            "run": run_dir.name,
            "model": run_data["model"],
            "time": record[TIME_FIELD],
            "cpu_recorded": recorded_joules,
            "cpu": record[CPU_FIELD],
            "dram": record[DRAM_FIELD],
            "gpu": record[GPU_FIELD],
        })
    run_rows.append({
        "run": run_dir.name,
        "inferences": len(records),
        "cpu_reference_watts": powers["cpu"],
        "dram_reference_watts": powers["dram"],
        "recorded_cpu_joules": recorded["cpu"],
        "recorded_total_joules": recorded["total"],
        **summary,
    })

    # 4. Write the files that changed
    changed_files = [path for path, file_records in files.items() if json.dumps(file_records) != on_disk[path]]
    files_changed += len(changed_files)
    summary_changed = json.dumps(summary) != json.dumps(run_data)
    summaries_changed += summary_changed

    if APPLY:
        for path in changed_files:
            with open(path, "w") as f:
                json.dump(files[path], f)
        if summary_changed:
            with open(run_dir / "run_data.json", "w") as f:
                json.dump(summary, f, indent=4)

inferences = pd.DataFrame(inference_rows)
runs = pd.DataFrame(run_rows)
overflow_log = pd.DataFrame(overflow_log)

print(f"Runs processed: {len(runs)}   inferences: {len(inferences)}")
print(f"Per-prompt files that {'were' if APPLY else 'would be'} rewritten: {files_changed}")
print(f"run_data.json files that {'were' if APPLY else 'would be'} rewritten: {summaries_changed}")

Runs processed: 530   inferences: 197160
Per-prompt files that were rewritten: 1905
run_data.json files that were rewritten: 530


In [6]:
# Run table: the columns created by Experiment Runner, followed by the fields of the summary
run_table_path = RESULTS_DIR / "run_table.csv"
with open(run_table_path, "r", newline="") as f:
    reader = csv.DictReader(f)
    previous_fieldnames = list(reader.fieldnames)
    rows = list(reader)

runner_columns = ["__run_id", "__done", "model", "benchmark", "runtime_seconds"]
summary_columns = [key for key in next(iter(new_summaries.values())) if key != "model"]
fieldnames = runner_columns + summary_columns

new_rows = []
for row in rows:
    summary = new_summaries.get(row["__run_id"])
    if summary is None:
        new_rows.append({column: row.get(column, "") for column in fieldnames})
        continue
    new_row = {column: row[column] for column in runner_columns}
    new_row.update({column: "" if summary[column] is None else str(summary[column]) for column in summary_columns})
    new_rows.append(new_row)

rows_changed = sum(new_row != {column: row.get(column) for column in fieldnames} for row, new_row in zip(rows, new_rows))

if APPLY:
    if rows_changed or fieldnames != previous_fieldnames:
        with open(run_table_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(new_rows)
    overflow_log.to_csv(RESULTS_DIR / "rapl_corrections.csv", index=False)

print(f"run_table.csv rows that {'were' if APPLY else 'would be'} updated: {rows_changed} of {len(rows)}")
if not APPLY:
    print("\nAPPLY is False: nothing was written.")

run_table.csv rows that were updated: 530 of 530


### 5. Validation

#### Counter Overflow Correction
We check the reference power of the runs, the distance of each corrected value to its expected energy (as a fraction of $M$; a wrong $k$ would give about 1) and the power of every inference after the correction.

In [7]:
print("Reference power of the runs (W):")
print(runs[["cpu_reference_watts", "dram_reference_watts"]].describe().loc[["min", "50%", "max"]].round(2).to_string())

print(f"\nInferences corrected: {overflow_log.groupby(['run', 'question_id']).ngroups} of {len(inferences)}, in {overflow_log['run'].nunique()} of {len(runs)} runs")
print("\nNumber of values by lost multiples of M (k = -1: overflow during a short inference):")
print(overflow_log.groupby(["domain", "k"]).size().unstack(fill_value=0).to_string())

print("\nDistance of the corrected values to their expected energy, as a fraction of M:")
print(overflow_log["error_fraction"].describe().loc[["50%", "max"]].round(3).to_string())

timed = inferences[inferences["time"] >= 1]
power = pd.DataFrame({
    "recorded": timed["cpu_recorded"] / timed["time"],
    "corrected": timed["cpu"] / timed["time"],
})
print("\nPower of the first processor in each inference (W):")
print(power.describe().loc[["min", "50%", "max"]].round(1).to_string())

Reference power of the runs (W):
     cpu_reference_watts  dram_reference_watts
min                86.50                 11.81
50%                87.36                 11.96
max                98.89                 12.10

Inferences corrected: 24250 of 197160, in 228 of 530 runs

Number of values by lost multiples of M (k = -1: overflow during a short inference):
k       -1      1     2     3     4    5
domain                                  
cpu     17  14151  6571  1090  1836  568
dram    17      0     0     0     0    0

Distance of the corrected values to their expected energy, as a fraction of M:
50%    0.001
max    0.122

Power of the first processor in each inference (W):
     recorded  corrected
min       0.0       10.2
50%      87.2       87.4
max    4159.5      104.2


In [8]:
# Overflows that could not be detected leave a value about 150 J too low, at most once per counter range consumed
undetected = runs["total_inference_cpu_joules"] * 1e6 / MAX_ENERGY_RANGE_UJ
bound_joules = undetected.apply(math.ceil) * R_MOD_M / 1e6
print(f"Largest possible undetected error in a run: {bound_joules.max():.0f} J "
      f"({100 * (bound_joules / runs['total_inference_joules']).max():.3f}% of its total measured energy)")

Largest possible undetected error in a run: 2556 J (0.166% of its total measured energy)


#### Effect on the Total Measured Energy
Change of the total measured energy of each model with the correction.

In [9]:
per_model = runs.groupby("model").agg(
    recorded=("recorded_total_joules", "sum"),
    corrected=("total_inference_joules", "sum"),
    gpu=("total_inference_gpu_joules", "sum"),
)
effect = pd.DataFrame({
    "change of the total (%)": 100 * (per_model["corrected"] / per_model["recorded"] - 1),
    "GPU share of the corrected total (%)": 100 * per_model["gpu"] / per_model["corrected"],
}).sort_values("change of the total (%)", ascending=False)

total_recorded, total_corrected = per_model["recorded"].sum(), per_model["corrected"].sum()
print(f"Total measured energy of the experiment: {total_recorded / 1e6:.0f} MJ recorded -> {total_corrected / 1e6:.0f} MJ corrected (+{100 * (total_corrected / total_recorded - 1):.1f}%)")
effect.round(1)

Total measured energy of the experiment: 1443 MJ recorded -> 1618 MJ corrected (+12.1%)


,change of the total (%),GPU share of the corrected total (%)
model,,
mistralai/Ministral-3-14B-Reasoning-2512,21.1,75.3
microsoft/Phi-4-reasoning,19.6,75.4
deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,17.5,75.3
Qwen/Qwen3-14B (Thinking Mode),16.9,75.3
Qwen/Qwen3.5-9B (Thinking Mode),16.9,75.0
deepseek-ai/DeepSeek-R1-0528-Qwen3-8B,14.8,75.2
mistralai/Ministral-3-8B-Reasoning-2512,14.0,75.2
Qwen/Qwen3-8B (Thinking Mode),13.9,75.1
Qwen/Qwen3.5-4B (Thinking Mode),12.6,74.7


#### GPU Energy vs. Total Measured Energy
Ranking of the models by accuracy per joule, using the GPU energy and using the total measured energy (computed here for this comparison only).

In [10]:
# Accuracy per joule of the total measured energy, defined as accuracy_per_gpu_joule is for the GPU energy
runs["accuracy_per_total_joule"] = runs["avg_accuracy"] / (runs["total_inference_joules"] / runs["inferences"])

by_model = runs.groupby("model")[["accuracy_per_gpu_joule", "accuracy_per_total_joule"]].mean()
gpu_rank = by_model["accuracy_per_gpu_joule"].rank(ascending=False)
total_rank = by_model["accuracy_per_total_joule"].rank(ascending=False)
difference = (gpu_rank - total_rank).abs()

print(f"Models: {len(by_model)}")
print(f"Models in a different position: {int((difference > 0).sum())}")
print(f"Largest change of position: {int(difference.max())}")
print(f"Correlation between the two rankings: {gpu_rank.corr(total_rank):.4f}")

# Power of the first processor and of its DRAM during inference, per model
power = runs.groupby("model")[["avg_cpu_power_watts", "avg_dram_power_watts"]].mean()
print("\nAverage power per model (W):")
print(power.agg(["min", "median", "max"]).round(1).to_string())

Models: 53
Models in a different position: 12
Largest change of position: 2
Correlation between the two rankings: 0.9993

Average power per model (W):
        avg_cpu_power_watts  avg_dram_power_watts
min                    86.5                  11.9
median                 87.3                  12.0
max                    89.3                  12.0


#### Recomputed Summaries vs. Recorded Summaries
For the fields that depend neither on the CPU and DRAM energy nor on the revised definitions, we compare the summary that was on disk with the recomputed one, and list the fields removed and added by this execution.

In [11]:
UNCHANGED_FIELDS = [
    "avg_accuracy", "total_inference_gpu_joules", "avg_gpu_power_watts",
    "total_tokens", "total_prompt_tokens", "total_generated_tokens", "total_thinking_tokens",
    "avg_prefill_tokens_per_second", "avg_decode_tokens_per_second",
    "avg_time_per_inference_seconds", "total_inference_time_seconds",
    "total_prefill_time_seconds", "total_decode_time_seconds",
    "avg_gpu_temperature_celsius", "avg_cpu_temperature_celsius",
]


def relative_difference(a, b):
    return 0.0 if a == b else abs(a - b) / max(abs(a), abs(b))


largest = max(
    relative_difference(previous[field], new_summaries[run][field])
    for run, previous in previous_summaries.items()
    for field in UNCHANGED_FIELDS
)
previous_fields = set().union(*previous_summaries.values())
new_fields = list(next(iter(new_summaries.values())))

print(f"Fields compared: {len(UNCHANGED_FIELDS)}, in {len(previous_summaries)} runs")
print(f"Largest relative difference: {largest:.1e}")
print(f"\nFields removed from the summaries on disk: {sorted(previous_fields - set(new_fields)) or 'none'}")
print(f"Fields added to the summaries on disk: {[field for field in new_fields if field not in previous_fields] or 'none'}")

Fields compared: 15, in 530 runs
Largest relative difference: 4.3e-16

Fields removed from the summaries on disk: ['accuracy_per_joule', 'avg_joules_per_inference', 'joules_per_token', 'tokens_per_joule']
Fields added to the summaries on disk: ['accuracy_per_gpu_joule', 'gpu_joules_per_correct_answer', 'generated_tokens_per_gpu_joule', 'gpu_joules_per_generated_token', 'avg_gpu_joules_per_inference', 'median_gpu_joules_per_inference', 'avg_generated_tokens_per_inference', 'median_generated_tokens_per_inference', 'truncated_responses', 'unparsed_responses', 'median_time_per_inference_seconds']
